### Zadanie 1 - PostgreSQL (lub SQLite)

Korzystajac z **Random User API** (`https://randomuser.me/api/?results=30`), wygeneruj 30 uzytkownikow i zapisz ich dane (imie, nazwisko, email, adres, wiek, plec) w tabeli SQL.

Nastepnie sprawdz SQL-em:
- Ile jest mezczyzn, a ile kobiet?
- Jaki jest sredni wiek?
- W ilu krajach mieszkaja?

Wzorujemy sie na notebooku Lab4 - uzywamy SQLite (PEP 249, te same 4 koncepty co psycopg).

In [1]:
import sqlite3
import requests

# 1. Pobierz dane z API
response = requests.get("https://randomuser.me/api/?results=30")
users = response.json()["results"]
print(f"Pobrano {len(users)} uzytkownikow.")


Pobrano 30 uzytkownikow.


In [2]:
# 2. Connection + cursor + CREATE TABLE
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.execute("""
    CREATE TABLE Users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        first_name TEXT,
        last_name TEXT,
        email TEXT,
        age INTEGER,
        gender TEXT,
        country TEXT,
        city TEXT
    )
""")
print("Tabela Users utworzona.")


Tabela Users utworzona.


In [3]:
# 3. INSERT z parametryzacja (? nie f-string!)
for u in users:
    cur.execute(
        "INSERT INTO Users (first_name, last_name, email, age, gender, country, city) VALUES (?, ?, ?, ?, ?, ?, ?)",
        (
            u["name"]["first"],
            u["name"]["last"],
            u["email"],
            u["dob"]["age"],
            u["gender"],
            u["location"]["country"],
            u["location"]["city"],
        )
    )
conn.commit()
print(f"Wstawiono {cur.execute('SELECT COUNT(*) FROM Users').fetchone()[0]} rekordow.")


Wstawiono 30 rekordow.


In [4]:
# 4. Zapytania analityczne

print("=== Liczba mezczyzn vs kobiet ===")
cur.execute("SELECT gender, COUNT(*) FROM Users GROUP BY gender")
for row in cur.fetchall():
    print(f"  {row[0]}: {row[1]}")

print("\n=== Sredni wiek ===")
cur.execute("SELECT ROUND(AVG(age), 1) FROM Users")
print(f"  {cur.fetchone()[0]} lat")

print("\n=== Liczba unikalnych krajow ===")
cur.execute("SELECT COUNT(DISTINCT country) FROM Users")
print(f"  {cur.fetchone()[0]} krajow")

print("\n=== Top 5 krajow ===")
cur.execute("""
    SELECT country, COUNT(*) as ile
    FROM Users
    GROUP BY country
    ORDER BY ile DESC
    LIMIT 5
""")
for row in cur.fetchall():
    print(f"  {row[0]}: {row[1]}")

conn.close()


=== Liczba mezczyzn vs kobiet ===
  female: 9
  male: 21

=== Sredni wiek ===
  55.3 lat

=== Liczba unikalnych krajow ===
  17 krajow

=== Top 5 krajow ===
  Turkey: 3
  Switzerland: 3
  Ireland: 3
  Australia: 3
  United States: 2
